In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("pyyaml")
install("scikit-learn")
install("tqdm")
install("matplotlib")
install("pandas")
install("git+https://github.com/openai/CLIP.git")

import clip
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

REPO_URL = "https://github.com/dvydinh/ralm_industrial_anomaly_detection.git"
REPO_DIR = "/content/ralm_industrial_anomaly_detection"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

sys.path.insert(0, REPO_DIR)

from raml.models.raml_model import RAMLModel
from raml.losses.combined_loss import MACCLLoss
from raml.data.mvtec_dataset import MVTecDataset
from raml.utils.metrics import compute_per_category_metrics



In [ ]:
import os
import subprocess
MVTEC_ROOT = "/content/drive/MyDrive/ralm/data/mvtec_anomaly_detection"
SAVE_DIR = "/content/drive/MyDrive/ralm"
for d in ["models", "metrics", "plots"]:
    os.makedirs(os.path.join(SAVE_DIR, d), exist_ok=True)
if not os.path.exists(os.path.join(MVTEC_ROOT, "bottle")):
    import kagglehub
    import shutil
    path = kagglehub.dataset_download("ipythonx/mvtec-ad")
    os.makedirs(MVTEC_ROOT, exist_ok=True)
    shutil.copytree(path, MVTEC_ROOT, dirs_exist_ok=True)
else:
    print(f"Dataset already exists at {MVTEC_ROOT}")


In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import json
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model, preprocess = clip.load("ViT-B/16", device=device)
tokenizer = clip.tokenize

test_ds = MVTecDataset(
    MVTEC_ROOT,
    categories=[d for d in os.listdir(MVTEC_ROOT) if os.path.isdir(os.path.join(MVTEC_ROOT, d))],
    split="test",
    transform=preprocess,
    use_test_anomalies=True,
    train_ratio=0.8,
    seed=42,
)
test_loader = DataLoader(
    test_ds, batch_size=8, shuffle=False, 
    num_workers=2, pin_memory=True, persistent_workers=True
)
print(f"WinCLIP Eval: {len(test_ds)} test samples")



In [ ]:
print("WinCLIP Zero-Shot Baseline (same backbone, same split)")
print("=" * 60)

clip_model.eval()
text_cache = {}

def winclip_score(image_feat, category):
    if category not in text_cache:
        with torch.no_grad():
            n_tok = tokenizer([f"a photo of a flawless {category}"]).to(device)
            a_tok = tokenizer([f"a photo of a damaged {category}"]).to(device)
            n_emb = F.normalize(clip_model.encode_text(n_tok).float(), dim=-1)
            a_emb = F.normalize(clip_model.encode_text(a_tok).float(), dim=-1)
            text_cache[category] = torch.cat([n_emb, a_emb], dim=0)
    tw = text_cache[category]
    logits = image_feat @ tw.t()
    return logits.softmax(dim=-1)[0, 1].item()

wc_labels, wc_scores, wc_cats = [], [], []
with torch.no_grad():
    for images, labels, cats in tqdm(test_loader, desc="WinCLIP"):
        images = images.to(device)
        feats = F.normalize(clip_model.encode_image(images).float(), dim=-1)
        for i in range(images.size(0)):
            wc_scores.append(winclip_score(feats[i:i+1], cats[i]))
            wc_labels.append(labels[i].item())
            wc_cats.append(cats[i])

wc_by_cat, wc_macro = compute_per_category_metrics(wc_cats, wc_labels, wc_scores)
wc_mean = wc_macro['auroc']

print(f"\nWinCLIP Mean AUROC: {wc_macro['auroc']*100:.2f}% | AP: {wc_macro['ap']*100:.2f}% | F1: {wc_macro['f1_max']*100:.2f}%")
for cat, m in sorted(wc_by_cat.items()):
    print(f"  {cat:15s}: AUROC {m['auroc']*100:.2f}% | AP {m['ap']*100:.2f}% | P {m['precision']*100:.2f}% | R {m['recall']*100:.2f}%")



In [ ]:
# Save JSON & CSV
wc_result = {
    "method": "WinCLIP_ZeroShot",
    "backbone": "ViT-B/16",
    "mean_auroc": round(wc_mean * 100, 2),
    "macro_metrics": {k: round(v * 100, 2) for k, v in wc_macro.items()},
    "per_category": {k: {m_k: round(m_v * 100, 2) for m_k, m_v in v.items()} for k, v in wc_by_cat.items()},
}
json_path = os.path.join(SAVE_DIR, "metrics", "result_global_clip.json")
with open(json_path, "w") as f:
    json.dump(wc_result, f, indent=2)

csv_path = os.path.join(SAVE_DIR, "metrics", "result_global_clip.csv")
rows = [{"Category": k, **{m: v[m] * 100 for m in wc_macro.keys()}} for k, v in wc_by_cat.items()]
df = pd.DataFrame(rows)
df.loc[len(df)] = ["MEAN", *[wc_macro[m] * 100 for m in wc_macro.keys()]]
df.to_csv(csv_path, index=False)
print(f"Global CLIP results strictly saved to {SAVE_DIR}/metrics/")

